## Questão 7 - Previsão de demanda

### Cenário

O Sr. Almir está furioso. No último verão, o estoque de "Coletes Salva-Vidas' acabou em 10 dias, e a
empresa perdeu milhares de reais em vendas. Por outro lado, compraram 'Ancoras" demais e elas
estão enferrujando no galpão. 

Gabriel Santos, o Tech Lead, disse que não da mais para confiar no *feeling*. Ele quer um modelo preditivo que diga exatamente quantas unidades venderemos no próximo mês para ajustar as compras com fornecedores.

### Premissas obrigatórias:

- O período de treino deve incluir dados **até** 31/12/2023.
- O período de teste deve ser todo o mês de Janeiro de 2024.
- A previsão deve ser feita em base diária.
- Não é permitido utilizar dados futuros no treino (data leakage).
- Considere apenas o produto: "Motor de Popa Yamaha Evo Dash 155HP"

### Tarefa:

1. Utilize o dataset `vendas_2023_2024.csv`
2. Construa um modelo baseline simples, utilizando: 
    * **Média móvel dos últimos 7 dias de vendas** (considerando apenas dados anteriores à data prevista).
3. Gere a previsão diária de vendas para Janeiro de 2024.
4. Compare as previsões com os valores reais do período de teste utilizando a métrica: `MAE - Mean Absolute Error`
5. Responda objetivamente:
    * O baseline é adequado para esse produto?
    * Cite uma limitação desse método.

In [3]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error

In [6]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error

# Carregar dados
vendas = pd.read_csv('../datasets/vendas_2023_2024.csv')

# Filtrar pelo produto específico (id_product = 54)
produto_id = 54
produto_nome = "Motor de Popa Yamaha Evo Dash 155HP"
vendas_produto = vendas[vendas['id_product'] == produto_id].copy()

# Padronizar coluna de data
vendas_produto['sale_date'] = vendas_produto['sale_date'].str.replace('/', '-')
vendas_produto['date'] = pd.to_datetime(vendas_produto['sale_date'], format='mixed', dayfirst=True)

# Agrupar por dia e somar quantidade
vendas_diarias = vendas_produto.groupby('date')['qtd'].sum().reset_index()

# Definir período de treino e teste
treino = vendas_diarias[vendas_diarias['date'] <= '2023-12-31']
teste = vendas_diarias[(vendas_diarias['date'] >= '2024-01-01') & (vendas_diarias['date'] <= '2024-01-31')]

print(f"Produto: {produto_nome} (ID: {produto_id})")
print(f"Dados de treino: {len(treino)} dias (até 31/12/2023)")
print(f"Dados de teste: {len(teste)} dias (janeiro 2024)")
print(f"Total vendas no período: {vendas_produto['qtd'].sum()} unidades")

Produto: Motor de Popa Yamaha Evo Dash 155HP (ID: 54)
Dados de treino: 28 dias (até 31/12/2023)
Dados de teste: 3 dias (janeiro 2024)
Total vendas no período: 556 unidades


In [2]:
%pip install scikit-learn

   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   -------------------------------------- - 7.9/8.1 MB 51.0 MB/s eta 0:00:01
   ---------------------------------------- 8.1/8.1 MB 32.7 MB/s  0:00:00
   ---------------------------------------- 0.0/37.3 MB ? eta -:--:--
   ----------- ---------------------------- 10.7/37.3 MB 52.9 MB/s eta 0:00:01
   ----------------- ---------------------- 16.0/37.3 MB 46.6 MB/s eta 0:00:01
   ----------------------- ---------------- 22.3/37.3 MB 36.1 MB/s eta 0:00:01
   ---------------------------------- ----- 32.5/37.3 MB 40.0 MB/s eta 0:00:01
   ---------------------------------------  37.2/37.3 MB 40.9 MB/s eta 0:00:01
   ---------------------------------------- 37.3/37.3 MB 30.2 MB/s  0:00:01

   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- 

In [7]:
# Implementar baseline: média móvel de 7 dias
def previsao_baseline(data_treino, data_teste):
    previsoes = []
    for i, row in data_teste.iterrows():
        data_atual = row['date']
        # Pegar os últimos 7 dias anteriores à data atual
        ultimos_7_dias = data_treino[data_treino['date'] < data_atual].tail(7)
        if len(ultimos_7_dias) > 0:
            media = ultimos_7_dias['qtd'].mean()
        else:
            media = 0  # Caso não haja dados suficientes
        previsoes.append(media)
    return previsoes

# Gerar previsões para janeiro 2024
previsoes = previsao_baseline(treino, teste)

# Adicionar previsões ao dataframe de teste
teste = teste.copy()
teste['previsao'] = previsoes

print("Previsões geradas:")
print(teste[['date', 'qtd', 'previsao']].head(10))

Previsões geradas:
         date  qtd   previsao
28 2024-01-05   10  11.142857
29 2024-01-21   11  11.142857
30 2024-01-22    6  11.142857


In [8]:
# Calcular MAE
mae = mean_absolute_error(teste['qtd'], teste['previsao'])
print(f"MAE (Mean Absolute Error): {mae:.2f}")

# Estatísticas adicionais
print(f"\nVendas reais médias em janeiro: {teste['qtd'].mean():.2f}")
print(f"Previsões médias: {teste['previsao'].mean():.2f}")
print(f"Total vendas reais janeiro: {teste['qtd'].sum()}")
print(f"Total previsões janeiro: {teste['previsao'].sum():.0f}")

MAE (Mean Absolute Error): 2.14

Vendas reais médias em janeiro: 9.00
Previsões médias: 11.14
Total vendas reais janeiro: 27
Total previsões janeiro: 33


## Respostas às perguntas

1. **O baseline é adequado para esse produto?**  
   Não, o MAE de 2.14 indica um erro médio significativo, especialmente considerando que as vendas diárias são baixas (média de 9 unidades/dia). O método não captura padrões sazonais ou tendências, resultando em previsões imprecisas para um produto com demanda variável.

2. **Cite uma limitação desse método.**  
   A média móvel simples não considera fatores externos como sazonalidade, feriados, condições climáticas ou promoções, que podem influenciar fortemente as vendas de produtos náuticos. Além disso, é reativo e não prevê mudanças de tendência.

In [1]:
import pandas as pd
import numpy as np


In [3]:
df = pd.read_csv('../datasets/vendas_2023_2024.csv')

In [28]:
# Recarregar os dados para reverter as mudanças
df = pd.read_csv('../datasets/vendas_2023_2024.csv')

# Função para parsear datas mistas (YYYY-MM-DD ou DD-MM-YYYY)
def parse_date(date_str):
    try:
        return pd.to_datetime(date_str, format='%Y-%m-%d')
    except ValueError:
        try:
            return pd.to_datetime(date_str, format='%d-%m-%Y')
        except ValueError:
            return pd.NaT

df['sale_date'] = df['sale_date'].apply(parse_date)

In [29]:
pd.concat([df.head(5), df.tail(5)])

,id,id_client,id_product,qtd,total,sale_date
0,0,42,105,11,3405.00,2023-09-10
1,1,3,136,9,16873.90,2024-09-15
2,2,25,139,7,9475.30,2024-08-13
3,4,20,23,5,55893.00,2023-02-03
4,5,8,57,4,451403.90,2024-02-12
9890,9995,30,139,6,8549.00,2023-03-11
9891,9996,9,111,7,28497.15,2023-09-17
9892,9997,38,123,2,5276.30,2023-06-20
9893,9998,33,97,6,771409.50,2024-10-23
9894,9999,8,20,6,185001.00,2023-06-15


In [30]:
produtos = pd.read_csv('../datasets/produtos_raw.csv')
resultado = produtos[produtos['name'].str.contains("155HP", case=False, na=False)]
print(resultado)

                                   name         price  code actual_category
54  Motor de Popa Yamaha Evo Dash 155HP  R$ 121534.82    54      Propulssão


In [31]:
# Filtrar o produto: "Motor de Polpa Yamaha Evo Dash 155HP"
id_produto = 54 
df_produto = df[df['id_product'] == id_produto].copy()

# Agrupar por data para garantir que temos a soma de vendas diárias
vendas_diarias = df_produto.groupby('sale_date')['qtd'].sum().reset_index()

# Criar um range completo de datas para não ignorar dias com zero vendas
datas_completas = pd.date_range(start=vendas_diarias['sale_date'].min(), end='2024-01-31')
vendas_diarias = vendas_diarias.set_index('sale_date').reindex(datas_completas, fill_value=0).reset_index()
vendas_diarias.columns = ['ds', 'y']

vendas_diarias.head()

,ds,y
0,2023-01-10,3
1,2023-01-11,0
2,2023-01-12,0
3,2023-01-13,0
4,2023-01-14,0


In [32]:
# Calcular a média móvel de 7 dias (Baseline)
vendas_diarias['previsao'] = vendas_diarias['y'].rolling(window=7).mean().shift(1) # O shift(1) é essencial para não usar o dado do próprio dia na previsão (evitar data leakage)

# Separar o período de teste: Janeiro 2024
jan2024 = vendas_diarias[(vendas_diarias['ds'] >= '2024-01-01') & (vendas_diarias['ds'] <= '2024-01-31')].copy()

# Calcular o MAE
jan2024['error_abs'] = np.abs(jan2024['y'] - jan2024['previsao'])
mae = jan2024['error_abs'].mean()

print(f"MAE para janeiro de 2024: {mae:.4f}")

# Visualização rápida da comparação
print("\nExemplo de previsões vs Real (Janeiro 2024):")
print(jan2024[['ds', 'y', 'previsao']].head(10))

MAE para janeiro de 2024: 0.9954

Exemplo de previsões vs Real (Janeiro 2024):
            ds  y  previsao
356 2024-01-01  0       0.0
357 2024-01-02  0       0.0
358 2024-01-03  0       0.0
359 2024-01-04  0       0.0
360 2024-01-05  0       0.0
361 2024-01-06  0       0.0
362 2024-01-07  0       0.0
363 2024-01-08  0       0.0
364 2024-01-09  0       0.0
365 2024-01-10  0       0.0


não deu certo sla

> Tarefa 1 - Preparar os dados brutos

In [34]:
import pandas as pd
import numpy as np

# --- leitura da base de vendas ---
# (aproveitamos o df_vendas já tratado nas etapas anteriores,
#  mas relemos aqui para deixar a questão autocontida)
df_vendas = pd.read_csv("../datasets/vendas_2023_2024.csv")

# garante que as datas estão como datetime
df_vendas["data_venda"] = pd.to_datetime(
    df_vendas["sale_date"], dayfirst=True, errors="coerce"
)

# -------------------------------------------------------------------
# filtro: apenas o produto alvo
# id_product = 54  →  "Motor de Popa Yamaha Evo Dash 155HP"
# -------------------------------------------------------------------
PRODUTO_ID   = 54
PRODUTO_NOME = "Motor de Popa Yamaha Evo Dash 155HP"

df_produto = df_vendas[df_vendas["id_product"] == PRODUTO_ID].copy()

print(f"Registros do produto: {len(df_produto)}")
print(df_produto[["data_venda", "qtd", "total"]].sort_values("data_venda").head())

Registros do produto: 62
     data_venda  qtd      total
6354 2023-04-11    4   486139.0
4666 2023-05-09    5   607674.0
2230 2023-07-04    7   808206.8
5641 2023-11-04   15  1731870.9
6098 2023-12-11   11  1336883.0


> Tarefa 2 — Construir a série diária com o calendário completo

In [35]:
# -------------------------------------------------------------------
# série diária: soma de unidades vendidas por dia
# usamos qtd (quantidade), não o valor — previsão de demanda = unidades
# -------------------------------------------------------------------
vendas_diarias = (
    df_produto
    .groupby("data_venda", as_index=False)["qtd"]
    .sum()
    .rename(columns={"qtd": "qtd_vendida"})
)

# --- calendário completo do período de treino até o fim do teste ---
# treino: até 31/12/2023 | teste: janeiro/2024
data_inicio = vendas_diarias["data_venda"].min()
data_fim_teste = pd.Timestamp("2024-01-31")

calendario = pd.DataFrame({
    "data_venda": pd.date_range(start=data_inicio, end=data_fim_teste, freq="D")
})

# LEFT JOIN — dias sem registro ficam com 0
serie = calendario.merge(vendas_diarias, on="data_venda", how="left")
serie["qtd_vendida"] = serie["qtd_vendida"].fillna(0).astype(int)

print(f"\nPeríodo completo: {serie['data_venda'].min().date()} → {serie['data_venda'].max().date()}")
print(f"Total de dias na série: {len(serie)}")
print(f"Dias com ao menos 1 venda: {(serie['qtd_vendida'] > 0).sum()}")
print(f"Dias sem venda (qtd = 0): {(serie['qtd_vendida'] == 0).sum()}")


Período completo: 2023-04-11 → 2024-01-31
Total de dias na série: 296
Dias com ao menos 1 venda: 6
Dias sem venda (qtd = 0): 290


> Tarefa 3 — Separar treino e teste

In [36]:
# -------------------------------------------------------------------
# corte treino / teste
# premissa: nenhum dado futuro pode contaminar o treino (data leakage)
# -------------------------------------------------------------------
CORTE = pd.Timestamp("2023-12-31")

df_treino = serie[serie["data_venda"] <= CORTE].copy().reset_index(drop=True)
df_teste  = serie[serie["data_venda"] >  CORTE].copy().reset_index(drop=True)

print(f"Treino: {df_treino['data_venda'].min().date()} → {df_treino['data_venda'].max().date()} ({len(df_treino)} dias)")
print(f"Teste:  {df_teste['data_venda'].min().date()} → {df_teste['data_venda'].max().date()} ({len(df_teste)} dias)")

Treino: 2023-04-11 → 2023-12-31 (265 dias)
Teste:  2024-01-01 → 2024-01-31 (31 dias)


> Tarefa 4 — Construir o baseline: Média Móvel dos últimos 7 dias

In [37]:
# -------------------------------------------------------------------
# baseline: média móvel de 7 dias
# para cada data de teste, a previsão = média dos 7 dias anteriores
# apenas dados do treino (ou dias de teste já "passados") são usados
# -------------------------------------------------------------------

# concatenamos treino + teste para facilitar a janela deslizante,
# mas a previsão só é calculada nas datas de teste
serie_completa = pd.concat([df_treino, df_teste], ignore_index=True)

JANELA = 7
previsoes = []

for idx, row in df_teste.iterrows():

    data_atual = row["data_venda"]

    # posição dessa data na série completa
    pos = serie_completa[serie_completa["data_venda"] == data_atual].index[0]

    # pega os 7 dias ANTERIORES (nunca inclui o dia atual)
    janela_dados = serie_completa.iloc[pos - JANELA : pos]["qtd_vendida"]

    previsao = janela_dados.mean()

    previsoes.append({
        "data_venda"  : data_atual,
        "qtd_real"    : row["qtd_vendida"],
        "qtd_prevista": round(previsao, 2),
    })

df_resultado = pd.DataFrame(previsoes)

print("\nPrevisão diária — Janeiro de 2024")
print(df_resultado.to_string(index=False))


Previsão diária — Janeiro de 2024
data_venda  qtd_real  qtd_prevista
2024-01-01         0          0.00
2024-01-02         0          0.00
2024-01-03         0          0.00
2024-01-04         0          0.00
2024-01-05        10          0.00
2024-01-06         0          1.43
2024-01-07         0          1.43
2024-01-08         0          1.43
2024-01-09         0          1.43
2024-01-10         0          1.43
2024-01-11         0          1.43
2024-01-12         0          1.43
2024-01-13         0          0.00
2024-01-14         0          0.00
2024-01-15         0          0.00
2024-01-16         0          0.00
2024-01-17         0          0.00
2024-01-18         0          0.00
2024-01-19         0          0.00
2024-01-20         0          0.00
2024-01-21         0          0.00
2024-01-22         0          0.00
2024-01-23         0          0.00
2024-01-24         0          0.00
2024-01-25         0          0.00
2024-01-26         0          0.00
2024-01-27         0

> Tarefa 5 — Avaliar com MAE

In [38]:
from sklearn.metrics import mean_absolute_error

# -------------------------------------------------------------------
# MAE: Mean Absolute Error
# interpretação direta: "em média, erramos X unidades por dia"
# -------------------------------------------------------------------
mae = mean_absolute_error(
    df_resultado["qtd_real"],
    df_resultado["qtd_prevista"]
)

print(f"\nMAE do baseline (Média Móvel 7 dias): {mae:.4f} unidades/dia")

# --- contexto para o MAE fazer sentido ---
media_real   = df_resultado["qtd_real"].mean()
max_real      = df_resultado["qtd_real"].max()
dias_com_venda = (df_resultado["qtd_real"] > 0).sum()

print(f"\nContexto do período de teste (Janeiro/2024):")
print(f"  Média diária real:   {media_real:.2f} unidades")
print(f"  Máximo diário real:  {max_real} unidades")
print(f"  Dias com venda:      {dias_com_venda} de {len(df_resultado)}")


MAE do baseline (Média Móvel 7 dias): 0.6455 unidades/dia

Contexto do período de teste (Janeiro/2024):
  Média diária real:   0.32 unidades
  Máximo diário real:  10 unidades
  Dias com venda:      1 de 31


> Resposta objetiva — Análise do baseline

In [39]:
# -------------------------------------------------------------------
# imprimimos a análise final de forma estruturada
# -------------------------------------------------------------------

print("=" * 60)
print("ANÁLISE DO BASELINE — Motor de Popa Yamaha Evo Dash 155HP")
print("=" * 60)

print(f"""
RESULTADO:
  MAE = {mae:.4f} unidades/dia
  Média real Janeiro/2024 = {media_real:.2f} unidades/dia

PERGUNTA 1 — O baseline é adequado para esse produto?

  Não. Este produto tem venda muito esparsa: a maior parte dos
  dias o registro de vendas é zero, com picos pontuais de volume
  alto (venda em lote ou pedido grande). Nesses casos, a média
  móvel tende a prever sempre próximo de zero — e erra feio nos
  dias em que o pedido aparece.

  Um MAE de {mae:.2f} unidades num produto cujo comportamento
  real é binário (vende muito ou não vende nada) indica que o
  modelo não captura a dinâmica real de demanda.

PERGUNTA 2 — Cite uma limitação desse método.

  A média móvel assume que o futuro se parece com os últimos
  7 dias — ela não detecta sazonalidade (ex: verão vende mais),
  não captura tendência de crescimento e não distingue dias úteis
  de fins de semana.

  Para produtos com demanda intermitente como este, um modelo
  mais adequado seria Croston's Method ou uma abordagem baseada
  em probabilidade de ocorrência + volume médio por pedido.
""")

ANÁLISE DO BASELINE — Motor de Popa Yamaha Evo Dash 155HP

RESULTADO:
  MAE = 0.6455 unidades/dia
  Média real Janeiro/2024 = 0.32 unidades/dia

PERGUNTA 1 — O baseline é adequado para esse produto?

  Não. Este produto tem venda muito esparsa: a maior parte dos
  dias o registro de vendas é zero, com picos pontuais de volume
  alto (venda em lote ou pedido grande). Nesses casos, a média
  móvel tende a prever sempre próximo de zero — e erra feio nos
  dias em que o pedido aparece.

  Um MAE de 0.65 unidades num produto cujo comportamento
  real é binário (vende muito ou não vende nada) indica que o
  modelo não captura a dinâmica real de demanda.

PERGUNTA 2 — Cite uma limitação desse método.

  A média móvel assume que o futuro se parece com os últimos
  7 dias — ela não detecta sazonalidade (ex: verão vende mais),
  não captura tendência de crescimento e não distingue dias úteis
  de fins de semana.

  Para produtos com demanda intermitente como este, um modelo
  mais adequado seria

In [40]:
# Consulta: Quantas vezes o produto 155HP (ID 54) foi comprado em 2023
produto_155hp_id = 54
produto_155hp_nome = "Motor de Popa Yamaha Evo Dash 155HP"

# Filtrar vendas do produto em 2023
vendas_155hp_2023 = df_vendas[
    (df_vendas['id_product'] == produto_155hp_id) & 
    (df_vendas['data_venda'].dt.year == 2023)
]

# Total de unidades vendidas
total_unidades_2023 = vendas_155hp_2023['qtd'].sum()

# Número de transações
num_transacoes_2023 = len(vendas_155hp_2023)

print(f"Produto: {produto_155hp_nome} (ID: {produto_155hp_id})")
print(f"Em 2023:")
print(f"  Número de transações: {num_transacoes_2023}")
print(f"  Total de unidades vendidas: {total_unidades_2023}")

Produto: Motor de Popa Yamaha Evo Dash 155HP (ID: 54)
Em 2023:
  Número de transações: 5
  Total de unidades vendidas: 42
